In [1]:
# =============================================================================
# Script 1 of 5 – Aspect Ratio
# =============================================================================
#
# Computes the aspect ratio of polygons: width / height of bounding rectangle.
#
# OUTPUT:
#   CSV: Aspect_Ratio_1.csv
#   LOG:   Aspect_Ratio_1.log
#   Columns: ROW_ID, Square_Meters, Aspect_Ratio_1
# =============================================================================

import sys, logging, time
import geopandas as gpd
import numpy as np

# USER SETTINGS
INPUT_PATH  = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg"
OUTPUT_PATH = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Aspect_Ratio_1.csv"
LOG_PATH    = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Aspect_Ratio_1.log"
ID_COL      = "ROW_ID"
AREA_COL    = "Square_Meters"

# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(LOG_PATH, encoding="utf-8")],
    force=True
)
log = logging.getLogger(__name__)

# HELPER FUNCTION
def aspect_ratio(geom):
    minx, miny, maxx, maxy = geom.bounds
    width = maxx - minx
    height = maxy - miny
    if height == 0: return np.nan
    return width / height

# MAIN FUNCTION
def compute_aspect_ratio(gdf):
    log.info("Computing aspect ratio for %d polygons...", len(gdf))
    t0 = time.perf_counter()
    gdf = gdf.copy()
    gdf["Aspect_Ratio_1"] = gdf.geometry.apply(aspect_ratio)
    elapsed = time.perf_counter() - t0
    log.info("Aspect ratio computed in %.2f seconds", elapsed)
    log.info("Stats -> mean=%.4f min=%.4f max=%.4f nulls=%d",
             float(gdf["Aspect_Ratio_1"].mean()),
             float(gdf["Aspect_Ratio_1"].min()),
             float(gdf["Aspect_Ratio_1"].max()),
             int(gdf["Aspect_Ratio_1"].isna().sum()))
    return gdf

# ENTRY POINT
if __name__ == "__main__":
    log.info("Loading dataset from %s ...", INPUT_PATH)
    gdf = gpd.read_file(INPUT_PATH)
    log.info("Loaded %d features (CRS: %s)", len(gdf), gdf.crs)
    
    for col in (ID_COL, AREA_COL):
        if col not in gdf.columns:
            raise KeyError(f"Missing required column: {col}")
    
    if gdf.crs and gdf.crs.is_geographic:
        log.warning("Geographic CRS detected (%s), reprojecting to EPSG:5070", gdf.crs)
        gdf = gdf.to_crs(epsg=5070)

    gdf["geometry"] = gdf.buffer(0)
    gdf = compute_aspect_ratio(gdf)
    
    output_cols = [ID_COL, AREA_COL, "Aspect_Ratio_1"]
    gdf[output_cols].to_csv(OUTPUT_PATH, index=False)
    log.info("CSV written to %s", OUTPUT_PATH)

2026-04-08 14:49:33,830 [INFO] Loading dataset from C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg ...
2026-04-08 14:49:34,397 [INFO] Loaded 19523 features (CRS: EPSG:5070)
2026-04-08 14:49:34,943 [INFO] Computing aspect ratio for 19523 polygons...
2026-04-08 14:49:35,058 [INFO] Aspect ratio computed in 0.10 seconds
2026-04-08 14:49:35,060 [INFO] Stats -> mean=2.3055 min=0.0141 max=113.8842 nulls=0
2026-04-08 14:49:35,104 [INFO] CSV written to C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Aspect_Ratio_1.csv


In [2]:
# =============================================================================
# Script 2 of 5 – Compactness (Circle-Based)
# =============================================================================
#
# CSV: Compactness_2.csv
# LOG:   Compactness_2.log
# Column: Compactness_2
# =============================================================================

import sys, logging, time
import geopandas as gpd
import numpy as np

INPUT_PATH  = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg"
OUTPUT_PATH = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Compactness_2.csv"
LOG_PATH    = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Compactness_2.log"
ID_COL      = "ROW_ID"
AREA_COL    = "Square_Meters"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(LOG_PATH, encoding="utf-8")],
    force=True
)
log = logging.getLogger(__name__)

def compute_compactness_circle(gdf):
    log.info("Computing circle-based compactness for %d polygons...", len(gdf))
    t0 = time.perf_counter()
    gdf = gdf.copy()
    area = gdf.geometry.area
    perimeter = gdf.geometry.length
    denom = 2.0 * np.sqrt(np.pi * area)
    gdf["Compactness_2"] = np.where(denom > 0, perimeter / denom, np.nan)
    elapsed = time.perf_counter() - t0
    log.info("Computed in %.2f seconds", elapsed)
    log.info("Stats -> mean=%.4f min=%.4f max=%.4f nulls=%d",
             float(gdf["Compactness_2"].mean()),
             float(gdf["Compactness_2"].min()),
             float(gdf["Compactness_2"].max()),
             int(gdf["Compactness_2"].isna().sum()))
    return gdf

if __name__ == "__main__":
    log.info("Loading dataset from %s ...", INPUT_PATH)
    gdf = gpd.read_file(INPUT_PATH)
    log.info("Loaded %d features (CRS: %s)", len(gdf), gdf.crs)

    for col in (ID_COL, AREA_COL):
        if col not in gdf.columns:
            raise KeyError(f"Missing required column: {col}")

    if gdf.crs and gdf.crs.is_geographic:
        log.warning("Geographic CRS detected (%s), reprojecting to EPSG:5070", gdf.crs)
        gdf = gdf.to_crs(epsg=5070)

    gdf["geometry"] = gdf.buffer(0)
    gdf = compute_compactness_circle(gdf)
    
    output_cols = [ID_COL, AREA_COL, "Compactness_2"]
    gdf[output_cols].to_csv(OUTPUT_PATH, index=False)
    log.info("CSV written to %s", OUTPUT_PATH)

2026-04-08 14:49:35,119 [INFO] Loading dataset from C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg ...
2026-04-08 14:49:35,688 [INFO] Loaded 19523 features (CRS: EPSG:5070)
2026-04-08 14:49:36,277 [INFO] Computing circle-based compactness for 19523 polygons...
2026-04-08 14:49:36,323 [INFO] Computed in 0.03 seconds
2026-04-08 14:49:36,323 [INFO] Stats -> mean=2.4054 min=1.0041 max=13.9452 nulls=0
2026-04-08 14:49:36,384 [INFO] CSV written to C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Compactness_2.csv


In [3]:
# =============================================================================
# Script 3 of 5 – Rectangularity Index
# =============================================================================
# CSV: Rectangularity_3.csv
# LOG:   Rectangularity_3.log
# Column: Rectangularity_3
# =============================================================================

import sys, logging, time
import geopandas as gpd
import numpy as np

INPUT_PATH  = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg"
OUTPUT_PATH = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Rectangularity_3.csv"
LOG_PATH    = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Rectangularity_3.log"
ID_COL      = "ROW_ID"
AREA_COL    = "Square_Meters"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(LOG_PATH, encoding="utf-8")],
    force=True
)
log = logging.getLogger(__name__)

def mbr_area(geom):
    return geom.minimum_rotated_rectangle.area

def compute_rectangularity(gdf):
    log.info("Computing rectangularity for %d polygons ...", len(gdf))
    t0 = time.perf_counter()
    gdf = gdf.copy()
    gdf["area_mbr"] = gdf.geometry.apply(mbr_area)
    gdf["Rectangularity_3"] = np.where(
        gdf["area_mbr"] > 0,
        gdf.geometry.area / gdf["area_mbr"],
        np.nan).clip(0,1)
    elapsed = time.perf_counter() - t0
    log.info("Computed in %.2f seconds", elapsed)
    log.info("Stats -> mean=%.4f min=%.4f max=%.4f nulls=%d",
             float(gdf["Rectangularity_3"].mean()),
             float(gdf["Rectangularity_3"].min()),
             float(gdf["Rectangularity_3"].max()),
             int(gdf["Rectangularity_3"].isna().sum()))
    return gdf

if __name__ == "__main__":
    log.info("Loading dataset from %s ...", INPUT_PATH)
    gdf = gpd.read_file(INPUT_PATH)
    log.info("Loaded %d features (CRS: %s)", len(gdf), gdf.crs)

    for col in (ID_COL, AREA_COL):
        if col not in gdf.columns:
            raise KeyError(f"Missing required column: {col}")

    if gdf.crs and gdf.crs.is_geographic:
        log.warning("Geographic CRS detected (%s), reprojecting to EPSG:5070", gdf.crs)
        gdf = gdf.to_crs(epsg=5070)

    gdf["geometry"] = gdf.buffer(0)
    gdf = compute_rectangularity(gdf)

    output_cols = [ID_COL, AREA_COL, "area_mbr", "Rectangularity_3"]
    gdf[output_cols].to_csv(OUTPUT_PATH, index=False)
    log.info("CSV written to %s", OUTPUT_PATH)

2026-04-08 14:49:36,400 [INFO] Loading dataset from C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg ...
2026-04-08 14:49:36,907 [INFO] Loaded 19523 features (CRS: EPSG:5070)
2026-04-08 14:49:37,440 [INFO] Computing rectangularity for 19523 polygons ...
2026-04-08 14:50:19,250 [INFO] Computed in 41.80 seconds
2026-04-08 14:50:19,252 [INFO] Stats -> mean=0.6146 min=0.0105 max=0.9991 nulls=0
2026-04-08 14:50:19,308 [INFO] CSV written to C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Rectangularity_3.csv


In [4]:
# =============================================================================
# Script 4 of 5 – Elongation Index
# =============================================================================
#
# CSV: Elongation_Index_4.csv
# LOG:   Elongation_Index_4.log
# Column: Elongation_Index_4
# =============================================================================

import sys, logging, time
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString

# USER SETTINGS
INPUT_PATH  = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg"
OUTPUT_PATH = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Elongation_Index_4.csv"
LOG_PATH    = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Elongation_Index_4.log"
ID_COL      = "ROW_ID"
AREA_COL    = "Square_Meters"
N_SAMPLE    = 200  # points for PCA

# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(LOG_PATH, encoding="utf-8")],
    force=True
)
log = logging.getLogger(__name__)

# HELPER
def _pca_axes(geom, n_pts=N_SAMPLE):
    coords = np.array(geom.exterior.coords)
    if len(coords) < n_pts:
        ls = LineString(coords)
        coords = np.array([[p.x, p.y] for p in [ls.interpolate(t, normalized=True) for t in np.linspace(0, 1, n_pts)]])
    coords -= coords.mean(axis=0)
    cov = np.cov(coords.T)
    eigvals = np.linalg.eigvalsh(cov)
    minor = 2.0 * np.sqrt(max(eigvals[0], 0))
    major = 2.0 * np.sqrt(max(eigvals[1], 0))
    return minor, major

# MAIN FUNCTION
def compute_elongation_index(gdf):
    log.info("Computing elongation index for %d polygons...", len(gdf))
    t0 = time.perf_counter()
    gdf = gdf.copy()
    axes = gdf.geometry.apply(_pca_axes)
    gdf["minor_axis"] = axes.apply(lambda x: x[0])
    gdf["major_axis"] = axes.apply(lambda x: x[1])
    gdf["Elongation_Index_4"] = np.where(
        gdf["major_axis"] > 0,
        1.0 - (gdf["minor_axis"] / gdf["major_axis"]),
        np.nan
    ).clip(0.0, 1.0)
    elapsed = time.perf_counter() - t0
    log.info("Elongation index computed in %.2f seconds", elapsed)
    log.info("Stats -> mean=%.4f min=%.4f max=%.4f nulls=%d",
             float(gdf["Elongation_Index_4"].mean()),
             float(gdf["Elongation_Index_4"].min()),
             float(gdf["Elongation_Index_4"].max()),
             int(gdf["Elongation_Index_4"].isna().sum()))
    return gdf

# ENTRY POINT
if __name__ == "__main__":
    log.info("Loading dataset from %s ...", INPUT_PATH)
    gdf = gpd.read_file(INPUT_PATH)
    log.info("Loaded %d features (CRS: %s)", len(gdf), gdf.crs)

    for col in (ID_COL, AREA_COL):
        if col not in gdf.columns:
            raise KeyError(f"Missing required column: {col}")

    if gdf.crs and gdf.crs.is_geographic:
        log.warning("Geographic CRS detected (%s), reprojecting to EPSG:5070", gdf.crs)
        gdf = gdf.to_crs(epsg=5070)

    gdf["geometry"] = gdf.buffer(0)
    gdf = compute_elongation_index(gdf)

    output_cols = [ID_COL, AREA_COL, "minor_axis", "major_axis", "Elongation_Index_4"]
    gdf[output_cols].to_csv(OUTPUT_PATH, index=False)
    log.info("CSV written to %s", OUTPUT_PATH)

2026-04-08 14:50:19,319 [INFO] Loading dataset from C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg ...
2026-04-08 14:50:19,901 [INFO] Loaded 19523 features (CRS: EPSG:5070)
2026-04-08 14:50:20,422 [INFO] Computing elongation index for 19523 polygons...
2026-04-08 14:50:55,165 [INFO] Elongation index computed in 34.75 seconds
2026-04-08 14:50:55,165 [INFO] Stats -> mean=0.7185 min=0.0068 max=0.9973 nulls=0
2026-04-08 14:50:55,255 [INFO] CSV written to C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Elongation_Index_4.csv


In [2]:
# This is the NLR Provided one which we used for the Level 2 Analysis! 

# =============================================================================
# Script 5 of 5 – Compactness (Polsby-Popper)
# =============================================================================
#
# CSV: Compactness_Polsby-Popper_5.csv
# LOG:   Compactness_Polsby-Popper_5.log
# Column: Compactness_Polsby-Popper_5
# =============================================================================

import sys, logging, time
import geopandas as gpd
import numpy as np

# USER SETTINGS
INPUT_PATH  = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Alaska\Alaska_Level_2_Analysis_AEP_GlintGlare_LCOE_Substations_TransmissionLines.gpkg"
OUTPUT_PATH = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Alaska\Alaska_Compactness_Polsby-Popper_5.csv"
LOG_PATH    = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Alaska\Alaska_Compactness_Polsby-Popper_5.log"
ID_COL      = "ROW_ID"
AREA_COL    = "Square_Meters"

# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(LOG_PATH, encoding="utf-8")],
    force=True
)
log = logging.getLogger(__name__)

# MAIN FUNCTION
def compute_shape_compactness(gdf):
    log.info("Computing Polsby-Popper compactness for %d polygons...", len(gdf))
    t0 = time.perf_counter()
    gdf = gdf.copy()
    area = gdf.geometry.area
    perimeter_sq = gdf.geometry.length ** 2
    gdf["Compactness_Polsby-Popper_5"] = np.where(
        perimeter_sq > 0,
        (4.0 * np.pi * area) / perimeter_sq,
        np.nan
    ).clip(0.0, 1.0)
    elapsed = time.perf_counter() - t0
    log.info("Computed in %.2f seconds", elapsed)
    log.info("Stats -> mean=%.4f min=%.4f max=%.4f nulls=%d",
             float(gdf["Compactness_Polsby-Popper_5"].mean()),
             float(gdf["Compactness_Polsby-Popper_5"].min()),
             float(gdf["Compactness_Polsby-Popper_5"].max()),
             int(gdf["Compactness_Polsby-Popper_5"].isna().sum()))
    return gdf

# ENTRY POINT
if __name__ == "__main__":
    log.info("Loading dataset from %s ...", INPUT_PATH)
    gdf = gpd.read_file(INPUT_PATH)
    log.info("Loaded %d features (CRS: %s)", len(gdf), gdf.crs)

    for col in (ID_COL, AREA_COL):
        if col not in gdf.columns:
            raise KeyError(f"Missing required column: {col}")

    if gdf.crs and gdf.crs.is_geographic:
        log.warning("Geographic CRS detected (%s), reprojecting to EPSG:5070", gdf.crs)
        gdf = gdf.to_crs(epsg=5070)

    gdf["geometry"] = gdf.buffer(0)
    gdf = compute_shape_compactness(gdf)

    output_cols = [ID_COL, AREA_COL, "Compactness_Polsby-Popper_5"]
    gdf[output_cols].to_csv(OUTPUT_PATH, index=False)
    log.info("CSV written to %s", OUTPUT_PATH)

2026-04-14 16:34:01,834 [INFO] Loading dataset from C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Alaska\Alaska_Level_2_Analysis_AEP_GlintGlare_LCOE_Substations_TransmissionLines.gpkg ...
2026-04-14 16:34:01,865 [INFO] Loaded 122 features (CRS: EPSG:3338)
2026-04-14 16:34:01,882 [INFO] Computing Polsby-Popper compactness for 122 polygons...
2026-04-14 16:34:01,882 [INFO] Computed in 0.00 seconds
2026-04-14 16:34:01,882 [INFO] Stats -> mean=0.2774 min=0.0116 max=0.8944 nulls=0
2026-04-14 16:34:01,897 [INFO] CSV written to C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\Alaska\Alaska_Compactness_Polsby-Popper_5.csv


In [6]:
# Below is the script to calculate all 5 shape indices and export as a single .csv

In [7]:
# =============================================================================
# All 5 Shape Indices – Unified Script
# =============================================================================
#
# Computes 5 polygon shape metrics for a single GeoPackage input:
# 1. Aspect Ratio (Axis ratio)
# 2. Circle-based Compactness
# 3. Rectangularity
# 4. Elongation Index (via PCA of boundary points)
# 5. Polsby-Popper Compactness (Isoperimetric Quotient)
#
# Exports a single CSV:
#   - ROW_ID
#   - Square_Meters
#   - Aspect_Ratio_1
#   - Compactness_2
#   - Rectangularity_3
#   - Elongation_Index_4
#   - Compactness_Polsby-Popper_5
#
# Log file name matches CSV
# =============================================================================

import sys
import logging
import time
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString

# =============================================================================
# USER SETTINGS
# =============================================================================

INPUT_PATH  = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg"
OUTPUT_CSV  = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\All_5_Shape_Indices.csv"
OUTPUT_LOG  = r"C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\All_5_Shape_Indices.log"

ID_COL      = "ROW_ID"
AREA_COL    = "Square_Meters"
N_SAMPLE    = 200  # boundary points for PCA

# =============================================================================
# LOGGING SETUP
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(OUTPUT_LOG, encoding="utf-8")
    ],
    force=True,
)
log = logging.getLogger(__name__)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def _pca_axes(geom, n_pts=N_SAMPLE):
    """Approximate minor and major axes for elongation and aspect ratio."""
    coords = np.array(geom.exterior.coords)
    if len(coords) < n_pts:
        ls = LineString(coords)
        coords = np.array([[p.x, p.y] for p in [ls.interpolate(t, normalized=True) for t in np.linspace(0, 1, n_pts)]])
    coords -= coords.mean(axis=0)
    cov = np.cov(coords.T)
    eigvals = np.linalg.eigvalsh(cov)
    minor = 2.0 * np.sqrt(max(eigvals[0], 0))
    major = 2.0 * np.sqrt(max(eigvals[1], 0))
    return minor, major

def mbr_area(geom):
    """Area of minimum bounding rectangle."""
    return geom.minimum_rotated_rectangle.area

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":

    log.info("Loading dataset from %s ...", INPUT_PATH)
    gdf = gpd.read_file(INPUT_PATH)
    log.info("Loaded %d features (CRS: %s)", len(gdf), gdf.crs)

    for col in (ID_COL, AREA_COL):
        if col not in gdf.columns:
            raise KeyError(f"Missing required column: {col}")

    if gdf.crs and gdf.crs.is_geographic:
        log.warning("Geographic CRS detected (%s) -- reprojecting to EPSG:5070", gdf.crs)
        gdf = gdf.to_crs(epsg=5070)

    # Ensure valid geometries
    gdf["geometry"] = gdf.buffer(0)

    t0 = time.perf_counter()

    # ----------------------
    # 1. Aspect Ratio (Axis ratio)
    # ----------------------
    axes = gdf.geometry.apply(_pca_axes)
    gdf["Aspect_Ratio_1"] = axes.apply(lambda x: x[1]/x[0] if x[0]>0 else np.nan)

    # ----------------------
    # 2. Circle-based Compactness
    # ----------------------
    area = gdf.geometry.area
    perimeter = gdf.geometry.length
    denom_circle = 2.0 * np.sqrt(np.pi * area)
    gdf["Compactness_2"] = np.where(denom_circle>0, perimeter/denom_circle, np.nan)

    # ----------------------
    # 3. Rectangularity
    # ----------------------
    gdf["Rectangularity_3"] = np.where(
        mbr_area_series := gdf.geometry.apply(mbr_area) > 0,
        gdf.geometry.area / gdf.geometry.apply(mbr_area),
        np.nan
    ).clip(0.0,1.0)

    # ----------------------
    # 4. Elongation Index
    # ----------------------
    gdf["minor_axis"], gdf["major_axis"] = zip(*axes)
    gdf["Elongation_Index_4"] = np.where(
        gdf["major_axis"]>0,
        1.0 - gdf["minor_axis"]/gdf["major_axis"],
        np.nan
    ).clip(0.0,1.0)

    # ----------------------
    # 5. Polsby-Popper Compactness
    # ----------------------
    gdf["Compactness_Polsby-Popper_5"] = np.where(
        perimeter**2>0,
        (4*np.pi*area)/(perimeter**2),
        np.nan
    ).clip(0.0,1.0)

    elapsed = time.perf_counter()-t0
    log.info("All 5 shape indices computed in %.1f s", elapsed)

    # ----------------------
    # Export CSV
    # ----------------------
    output_cols = [
        ID_COL,
        AREA_COL,
        "Aspect_Ratio_1",
        "Compactness_2",
        "Rectangularity_3",
        "Elongation_Index_4",
        "Compactness_Polsby-Popper_5"
    ]
    log.info("Writing %d rows to %s ...", len(gdf), OUTPUT_CSV)
    gdf[output_cols].to_csv(OUTPUT_CSV, index=False)
    log.info("Done. Output columns: %s", output_cols)

2026-04-08 14:50:56,454 [INFO] Loading dataset from C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\CONUS_Large_Scale_Top_150_Facilities.gpkg ...
2026-04-08 14:50:56,985 [INFO] Loaded 19523 features (CRS: EPSG:5070)
2026-04-08 14:52:53,574 [INFO] All 5 shape indices computed in 116.1 s
2026-04-08 14:52:53,581 [INFO] Writing 19523 rows to C:\Users\KyleSteen.AzureAD\Documents\Facility_Shape_Workspace\All_5_Shape_Indices.csv ...
2026-04-08 14:52:53,698 [INFO] Done. Output columns: ['ROW_ID', 'Square_Meters', 'Aspect_Ratio_1', 'Compactness_2', 'Rectangularity_3', 'Elongation_Index_4', 'Compactness_Polsby-Popper_5']
